# Income Pipeline, Part 2: Web Scraping (Highest-Income ZIP Codes)

**Goal:** scrape Wikipedia's list of the 100 U.S. ZIP code areas with the highest per capita income.

**Steps:** request the page (with a User-Agent header, which Wikipedia now requires), find the right table with BeautifulSoup, and clean it: remove commas and footnote marks, convert to numbers, and standardize the ZIP codes.

**Result:** today's table is scraped and saved as `Data/TopIncome_wiki_current.csv`. The merge in part 4 uses the July 2024 version of the table (`Data/TopIncome_wiki.csv`), which also named each place; only 50 of its 100 ZIP codes are still on today's list.

**Tools:** Python (requests, BeautifulSoup, pandas)

**Context:** master's coursework (DSC540 term project), cleaned up and corrected for this portfolio. This is part of a four-notebook pipeline: 1 flat file, 2 web scraping, 3 API, 4 merge and visualize.

## 1. Scrape the table from Wikipedia

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

url = 'https://en.wikipedia.org/wiki/List_of_highest-income_ZIP_Code_Tabulation_Areas_in_the_United_States'
# Wikipedia refuses requests that do not identify themselves, so a User-Agent header is sent
response = requests.get(url, headers={'User-Agent': 'PortfolioDataProject/1.0 (educational use)'}, timeout=30)
print('Status code:', response.status_code)
soup = BeautifulSoup(response.text, 'html.parser')

Status code: 200


In [2]:
# find the table whose header row includes ZCTA and Per Capita Income
def find_income_table(soup):
    for table in soup.find_all('table'):
        headers = [th.get_text(' ', strip=True) for th in table.find_all('th')]
        if any('ZCTA' in h for h in headers) and any('Per Capita' in h for h in headers):
            return table, headers
    return None, None

table, headers = find_income_table(soup)
rows = [[td.get_text(' ', strip=True) for td in tr.find_all('td')] for tr in table.find_all('tr')[1:]]
live = pd.DataFrame([r for r in rows if len(r) == len(headers)], columns=headers)
print(f'Scraped {len(live)} rows; columns: {headers}')
live.head()

Scraped 100 rows; columns: ['Rank', 'ZCTA', 'Population', 'Per Capita Income ( USD )']


,Rank,ZCTA,Population,Per Capita Income ( USD )
0,1,33109,"1,083","302,761"
1,2,11005,"2,174","218,922"
2,3,10007,"7,406","212,686"
3,4,10282,"5,887","205,317"
4,5,94105,"13,861","197,122"


## 2. Clean the scraped table

In [3]:
live.columns = ['Rank', 'ZipCode', 'Population', 'PerCapitaIncome']
for col in ['Rank', 'Population', 'PerCapitaIncome']:
    live[col] = pd.to_numeric(live[col].str.replace(r'[^0-9]', '', regex=True))   # remove commas and footnote marks
live['ZipCode'] = live['ZipCode'].str.extract(r'(\d{5})', expand=False)
print('Missing values:', live.isna().sum().sum(), '  Duplicate ZIP codes:', live['ZipCode'].duplicated().sum())
live.to_csv('Data/TopIncome_wiki_current.csv', index=False)
print(f"Saved Data/TopIncome_wiki_current.csv. Highest per capita income today: ZIP {live.iloc[0]['ZipCode']} (${live.iloc[0]['PerCapitaIncome']:,})")

Missing values: 0   Duplicate ZIP codes: 0
Saved Data/TopIncome_wiki_current.csv. Highest per capita income today: ZIP 33109 ($302,761)


## 3. The July 2024 snapshot used in the merge
When this project was built (July 2024), the Wikipedia table also named each place (for example "Montchanin, Delaware"). That version was scraped, cleaned (place names split into city and state, state names converted to abbreviations, ZIP codes padded to five digits) and saved as `Data/TopIncome_wiki.csv`. The page has since been updated with newer figures and no longer names the places, so the merge in notebook 4 uses the saved snapshot.

In [4]:
snapshot = pd.read_csv('Data/TopIncome_wiki.csv', dtype={'ZipCode': str})
snapshot['ZipCode'] = snapshot['ZipCode'].str.zfill(5)
print(f'Snapshot: {len(snapshot)} ZIP codes in {snapshot["State"].nunique()} states')
print(f"ZIP codes in both the snapshot and today's table: {snapshot['ZipCode'].isin(live['ZipCode']).sum()} of {len(snapshot)}")
snapshot.head()

Snapshot: 100 ZIP codes in 25 states
ZIP codes in both the snapshot and today's table: 50 of 100


,Rank,CityState,ZipCode,Population,Per CapitaIncome,City,State
0,1,Montchanin Delaware,19710,68,654485,Montchanin,DE
1,2,Houston Texas,77010,76,283189,Houston,TX
2,3,Rockland Delaware,19732,77,279424,Rockland,DE
3,4,Miami Beach Florida,33109,467,236238,Miami Beach,FL
4,5,Pineland Florida,33945,79,162075,Pineland,FL
